In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import os
import glob
import json
from shapely.geometry import shape, Point
from sklearn.model_selection import train_test_split

In [2]:
file_path = '../data_for_lstm/var_depths_data_for_LSTM_B4_icemask.nc'
ds = xr.open_dataset(file_path)
ds

<xarray.Dataset> Size: 2GB
Dimensions:            (profile: 183565, depth: 102)
Coordinates:
  * profile            (profile) int64 1MB 0 1 2 3 ... 183562 183563 183564
    x_ease             (profile) float64 1MB ...
    y_ease             (profile) float64 1MB ...
  * depth              (depth) float64 816B 0.0 5.0 10.0 ... 5.4e+03 5.5e+03
Data variables: (12/25)
    LATITUDE           (profile) float64 1MB ...
    LONGITUDE          (profile) float64 1MB ...
    TIME               (profile) datetime64[ns] 1MB ...
    TEMP               (profile, depth) float64 150MB ...
    PSAL               (profile, depth) float64 150MB ...
    PRES               (profile, depth) float64 150MB ...
    ...                 ...
    S_glorys           (profile, depth) float64 150MB ...
    T_glorys           (profile, depth) float64 150MB ...
    S_glorys_monthly   (profile, depth) float64 150MB ...
    T_glorys_monthly   (profile, depth) float64 150MB ...
    bathymetry         (profile) float64 1MB ...
    ice_conc           (profile) float32 734kB ...
Attributes:
    title:                  An Arctic Ocean Thermohaline Dataset
    institution:            Key Laboratory of Marine Hazard Forecasting, Mini...
    platform_type:          XX
    doi:                    10.1038/s41597-025-05855-3
    glorys_offset_pattern:  A tolerance window of 5 by 5 indices centered at ...

In [3]:
if 'day_of_year' not in ds:
    print("Not found. Adding 'day_of_year' variable to the dataset...")
    ds['day_of_year'] = xr.DataArray(
        ds['TIME'].dt.dayofyear.values.astype('int32'),
        dims='profile',
        attrs={
            'units': '1',
            'long_name': 'Day of the year',
            'description': 'Day of the year, ranging 1 to 366.'
        }
    )


Not found. Adding 'day_of_year' variable to the dataset...


In [4]:
print(f"Dataset dimensions: {ds.sizes}")
print(f"Dataset shape: {dict(ds.sizes)}")
print(f"Data variables: {list(ds.data_vars)}")
print(f"Coordinates: {list(ds.coords)}")

Dataset dimensions: Frozen({'profile': 183565, 'depth': 102})
Dataset shape: {'profile': 183565, 'depth': 102}
Data variables: ['LATITUDE', 'LONGITUDE', 'TIME', 'TEMP', 'PSAL', 'PRES', 'DENS', 'TEMP_augs', 'PSAL_augs', 'TEMP_aug_fraction', 'PSAL_aug_fraction', 'lstm_training_QC', 'X_EASE', 'Y_EASE', 'SST_glorys', 'SSS_glorys', 'ADT', 'SST', 'SSS', 'S_glorys', 'T_glorys', 'S_glorys_monthly', 'T_glorys_monthly', 'bathymetry', 'ice_conc', 'day_of_year']
Coordinates: ['depth', 'profile', 'x_ease', 'y_ease']


In [5]:
# Load arctic regions and assign each profile to a region for stratified splitting
geojson_path = '/home/nicolas/SACO/FRESH-CARE/Arctic_masks/geojson_masks/arctic_regions.json'
with open(geojson_path) as f:
    regions_geojson = json.load(f)

# Build shapely polygons for each region
region_polygons = []
for feature in regions_geojson['features']:
    region_polygons.append({
        'id': feature['properties']['id'],
        'name': feature['properties']['name_en'],
        'geometry': shape(feature['geometry'])
    })

# Assign each profile to a region
lats = ds['LATITUDE'].values
lons = ds['LONGITUDE'].values
region_labels = np.full(len(lats), 'Other', dtype=object)

for i in range(len(lats)):
    pt = Point(lons[i], lats[i])
    for reg in region_polygons:
        if reg['geometry'].contains(pt):
            region_labels[i] = reg['id']
            break

unique, counts = np.unique(region_labels, return_counts=True)
print("Profiles per region:")
for u, c in zip(unique, counts):
    print(f"  {u}: {c} ({c/len(lats):.2%})")

Profiles per region:
  Other: 5861 (3.19%)
  S1: 34666 (18.88%)
  S10: 23890 (13.01%)
  S11: 4492 (2.45%)
  S12: 4692 (2.56%)
  S13: 13873 (7.56%)
  S14: 3224 (1.76%)
  S2: 38804 (21.14%)
  S3: 6703 (3.65%)
  S4: 22 (0.01%)
  S5: 455 (0.25%)
  S6: 633 (0.34%)
  S7: 12701 (6.92%)
  S8: 31530 (17.18%)
  S9: 2019 (1.10%)


In [6]:
# Set random seed for reproducibility
random_seed = 42

# Get total number of profiles
n_profiles = ds.sizes['profile']
print(f"Total profiles: {n_profiles}")

train_size = 0.63
dev_size = 0.21
test_size = 0.16

# Stratified split: first separate train from (dev+test)
all_indices = np.arange(n_profiles)

indices_train, indices_devtest = train_test_split(
    all_indices,
    train_size=train_size,
    random_state=random_seed,
    stratify=region_labels
)

# Second split: separate dev and test from the remaining
# dev_size / (dev_size + test_size) gives the proportion of dev within devtest
dev_fraction = dev_size / (dev_size + test_size)
labels_devtest = region_labels[indices_devtest]

indices_dev, indices_test = train_test_split(
    indices_devtest,
    train_size=dev_fraction,
    random_state=random_seed,
    stratify=labels_devtest
)

print(f"train split: {len(indices_train)} profiles ({len(indices_train)/n_profiles:.4%})")
print(f"dev split:   {len(indices_dev)} profiles ({len(indices_dev)/n_profiles:.4%})")
print(f"test split:  {len(indices_test)} profiles ({len(indices_test)/n_profiles:.4%})")

# Verify regional proportions are preserved
print("\nRegion proportions (full -> train / dev / test):")
for reg in np.unique(region_labels):
    full_pct = np.mean(region_labels == reg) * 100
    train_pct = np.mean(region_labels[indices_train] == reg) * 100
    dev_pct = np.mean(region_labels[indices_dev] == reg) * 100
    test_pct = np.mean(region_labels[indices_test] == reg) * 100
    print(f"  {reg}: {full_pct:5.2f}% -> {train_pct:5.2f}% / {dev_pct:5.2f}% / {test_pct:5.2f}%")

Total profiles: 183565
train split: 115645 profiles (62.9995%)
dev split:   38549 profiles (21.0002%)
test split:  29371 profiles (16.0003%)

Region proportions (full -> train / dev / test):
  Other:  3.19% ->  3.19% /  3.19% /  3.19%
  S1: 18.88% -> 18.88% / 18.89% / 18.89%
  S10: 13.01% -> 13.01% / 13.01% / 13.01%
  S11:  2.45% ->  2.45% /  2.45% /  2.45%
  S12:  2.56% ->  2.56% /  2.56% /  2.56%
  S13:  7.56% ->  7.56% /  7.56% /  7.56%
  S14:  1.76% ->  1.76% /  1.76% /  1.76%
  S2: 21.14% -> 21.14% / 21.14% / 21.14%
  S3:  3.65% ->  3.65% /  3.65% /  3.65%
  S4:  0.01% ->  0.01% /  0.01% /  0.01%
  S5:  0.25% ->  0.25% /  0.25% /  0.25%
  S6:  0.34% ->  0.35% /  0.35% /  0.34%
  S7:  6.92% ->  6.92% /  6.92% /  6.92%
  S8: 17.18% -> 17.18% / 17.18% / 17.18%
  S9:  1.10% ->  1.10% /  1.10% /  1.10%


In [7]:
# Create subsets using isel and compute immediately
ds_train = ds.isel(profile=indices_train).compute()
ds_dev = ds.isel(profile=indices_dev).compute()
ds_test = ds.isel(profile=indices_test).compute()
print(f"Split complete: {ds_train.sizes['profile']} + " 
      f"{ds_dev.sizes['profile']} + "
      f"{ds_test.sizes['profile']} "
      f"= {ds.sizes['profile']}")

Split complete: 115645 + 38549 + 29371 = 183565


In [8]:
# Save datasets with suffix
output_train = '../data_for_lstm/var_depths_data_for_LSTM_C_wg_train63.nc'
output_dev = '../data_for_lstm/var_depths_data_for_LSTM_C_wg_dev21.nc'
output_test = '../data_for_lstm/var_depths_data_for_LSTM_C_wg_test16.nc'

ds_train.to_netcdf(output_train)
ds_dev.to_netcdf(output_dev)
ds_test.to_netcdf(output_test)

print(f"Saved: {output_train}")
print(f"Saved: {output_dev}")
print(f"Saved: {output_test}")

Saved: ../data_for_lstm/var_depths_data_for_LSTM_C_wg_train63.nc
Saved: ../data_for_lstm/var_depths_data_for_LSTM_C_wg_dev21.nc
Saved: ../data_for_lstm/var_depths_data_for_LSTM_C_wg_test16.nc


In [9]:
ds_train

<xarray.Dataset> Size: 959MB
Dimensions:            (profile: 115645, depth: 102)
Coordinates:
  * profile            (profile) int64 925kB 39099 6888 163162 ... 178059 104111
    x_ease             (profile) float64 925kB -9.847e+05 ... -4.247e+05
    y_ease             (profile) float64 925kB 1.259e+06 ... 2.825e+06
  * depth              (depth) float64 816B 0.0 5.0 10.0 ... 5.4e+03 5.5e+03
Data variables: (12/26)
    LATITUDE           (profile) float64 925kB 75.65 78.64 73.0 ... 74.63 64.19
    LONGITUDE          (profile) float64 925kB -142.0 10.82 ... -142.9 -171.4
    TIME               (profile) datetime64[ns] 925kB 2013-09-11T00:02:00.010...
    TEMP               (profile, depth) float64 94MB -1.509 -1.48 ... nan nan
    PSAL               (profile, depth) float64 94MB 27.24 27.29 ... nan nan
    PRES               (profile, depth) float64 94MB 0.0 0.0 10.11 ... 0.0 0.0
    ...                 ...
    T_glorys           (profile, depth) float64 94MB -1.543 -1.543 ... nan nan
    S_glorys_monthly   (profile, depth) float64 94MB 28.39 28.39 ... nan nan
    T_glorys_monthly   (profile, depth) float64 94MB -1.541 -1.541 ... nan nan
    bathymetry         (profile) float64 925kB -3.741e+03 140.4 ... -44.93
    ice_conc           (profile) float32 463kB 100.0 nan 0.0 ... 0.0 95.59 0.0
    day_of_year        (profile) int32 463kB 254 126 230 207 ... 268 234 297 354
Attributes:
    title:                  An Arctic Ocean Thermohaline Dataset
    institution:            Key Laboratory of Marine Hazard Forecasting, Mini...
    platform_type:          XX
    doi:                    10.1038/s41597-025-05855-3
    glorys_offset_pattern:  A tolerance window of 5 by 5 indices centered at ...